In [ ]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

PROJECT_DIR = Path(
    "/workspaces/DBA-ai-trust-score-framework-Interim"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"
REPORTS = PROJECT_DIR / "outputs"/"figures"

best_model = joblib.load(
    OUTPUT_DIR / "best_model.pkl"
)

print("Model loaded:", type(best_model).__name__)

In [ ]:
DATA = pd.read_csv(
    PROCESSED_DIR / "dms_features.csv"
)

print(DATA.shape)

In [ ]:
(1400, 18)

In [ ]:
from sklearn.preprocessing import LabelEncoder

TARGET = "label"

encoder = LabelEncoder()

DATA["target"] = encoder.fit_transform(
    DATA[TARGET]
)

MODEL_FEATURES = list(
    best_model.feature_names_in_
)

X = DATA[MODEL_FEATURES]
y = DATA["target"]

print("Features:", len(MODEL_FEATURES))
print(MODEL_FEATURES)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Test samples:", len(X_test))

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)

print("Predictions:", len(y_pred))
print("Probabilities:", y_prob.shape)

### KPI 1 — PERFORMANCE

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

performance_metrics = {

    "Accuracy": accuracy_score(
        y_test,
        y_pred
    ),

    "Precision": precision_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    ),

    "Recall": recall_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    ),

    "Macro_F1": f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )
}

performance_metrics

In [ ]:
performance_score = performance_metrics["Macro_F1"]

print(
    f"Performance KPI: {performance_score:.4f}"
)

Per Class Performance

In [ ]:
class_report = classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_,
    output_dict=True,
    zero_division=0
)

class_performance = pd.DataFrame(
    class_report
).T

class_performance

In [ ]:
class_performance.to_csv(
    OUTPUT_DIR / "class_performance.csv"
)

### KPI 2 — SAFETY

In [ ]:
UNSAFE_CLASSES = [
    "DangerousDriving",
    "Distracted",
    "Drinking",
    "Yawn"
]

SAFE_CLASS = "SafeDriving"

actual_labels = encoder.inverse_transform(
    y_test
)

predicted_labels = encoder.inverse_transform(
    y_pred
)

safety_df = pd.DataFrame({

    "actual": actual_labels,

    "predicted": predicted_labels

})

safety_df.head()

In [ ]:
critical_errors = (

    safety_df["actual"].isin(
        UNSAFE_CLASSES
    )

    &

    (safety_df["predicted"] == SAFE_CLASS)

)

critical_error_count = critical_errors.sum()

critical_error_rate = (
    critical_error_count
    /
    safety_df["actual"].isin(
        UNSAFE_CLASSES
    ).sum()
)

print(
    "Safety-critical errors:",
    critical_error_count
)

print(
    "Safety-critical error rate:",
    round(
        critical_error_rate,
        4
    )
)

In [ ]:
safety_score = 1 - critical_error_rate

In [ ]:
safety_score = max(
    0,
    1 - critical_error_rate
)

print(
    f"Safety KPI: {safety_score:.4f}"
)

### KPI 3 — ROBUSTNESS

In [ ]:
def evaluate_f1(
    model,
    X_data,
    y_true
):

    predictions = model.predict(
        X_data
    )

    return f1_score(
        y_true,
        predictions,
        average="macro",
        zero_division=0
    )

In [ ]:
baseline_f1 = evaluate_f1(
    best_model,
    X_test,
    y_test
)

print(
    "Baseline Macro-F1:",
    round(
        baseline_f1,
        4
    )
)

Brightness degradation

In [ ]:
X_brightness = X_test.copy()

X_brightness["brightness"] = (
    X_brightness["brightness"] * 0.60
)

brightness_f1 = evaluate_f1(
    best_model,
    X_brightness,
    y_test
)

print(
    "Brightness-degraded F1:",
    round(
        brightness_f1,
        4
    )
)

Contrast degradation

In [ ]:
X_contrast = X_test.copy()

X_contrast["contrast"] = (
    X_contrast["contrast"] * 0.60
)

contrast_f1 = evaluate_f1(
    best_model,
    X_contrast,
    y_test
)

print(
    "Contrast-degraded F1:",
    round(
        contrast_f1,
        4
    )
)

Blur degradation

In [ ]:
X_blur = X_test.copy()

X_blur["blur_score"] = (
    X_blur["blur_score"] * 0.40
)

X_blur["edge_density"] = (
    X_blur["edge_density"] * 0.70
)

blur_f1 = evaluate_f1(
    best_model,
    X_blur,
    y_test
)

print(
    "Blur-degraded F1:",
    round(
        blur_f1,
        4
    )
)

Robustness table

In [ ]:
robustness_results = pd.DataFrame({

    "Condition": [
        "Original",
        "Reduced Brightness",
        "Reduced Contrast",
        "Blurred"
    ],

    "Macro_F1": [
        baseline_f1,
        brightness_f1,
        contrast_f1,
        blur_f1
    ]

})

robustness_results

In [ ]:
degraded_scores = [
    brightness_f1,
    contrast_f1,
    blur_f1
]

robustness_score = (
    np.mean(degraded_scores)
    /
    baseline_f1
)

robustness_score = np.clip(
    robustness_score,
    0,
    1
)

print(
    f"Robustness KPI: {robustness_score:.4f}"
)

In [ ]:
degraded_mean_f1 = np.mean(degraded_scores)

robustness_retention = (
    degraded_mean_f1 / baseline_f1
)

performance_degradation = (
    1 - robustness_retention
)

print(f"Baseline Macro-F1:        {baseline_f1:.4f}")
print(f"Mean degraded Macro-F1:   {degraded_mean_f1:.4f}")
print(f"Performance retention:    {robustness_retention:.4f}")
print(f"Performance degradation:  {performance_degradation*100 :.1f}%")

### KPI 4 — Calibration

Multiclass Brier Score - This answers:

When the model says it is highly confident, can we trust that confidence? Lower the better

In [ ]:
from sklearn.preprocessing import label_binarize

classes = np.arange(
    len(encoder.classes_)
)

y_test_binary = label_binarize(
    y_test,
    classes=classes
)

brier_score = np.mean(
    np.sum(
        (
            y_prob
            -
            y_test_binary
        ) ** 2,
        axis=1
    )
)

print(
    f"Multiclass Brier Score: "
    f"{brier_score:.4f}"
)

In [ ]:
confidence = np.max(
    y_prob,
    axis=1
)

correct = (
    y_pred == y_test
)

n_bins = 10

bins = np.linspace(
    0,
    1,
    n_bins + 1
)

ece = 0.0

for i in range(n_bins):

    if i == n_bins - 1:

        mask = (
            (confidence >= bins[i])
            &
            (confidence <= bins[i + 1])
        )

    else:

        mask = (
            (confidence >= bins[i])
            &
            (confidence < bins[i + 1])
        )

    if mask.sum() == 0:
        continue

    bin_accuracy = correct[mask].mean()

    bin_confidence = confidence[mask].mean()

    bin_weight = mask.mean()

    ece += (
        bin_weight
        *
        abs(
            bin_accuracy
            -
            bin_confidence
        )
    )

print(
    f"Expected Calibration Error: "
    f"{ece:.4f}"
)

In [ ]:
calibration_score = np.clip(
    1 - ece,
    0,
    1
)

print(
    f"Calibration KPI: "
    f"{calibration_score:.4f}"
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.calibration import calibration_curve

# Correctness of prediction
correct = (
    y_pred == y_test
).astype(int)

confidence = np.max(
    y_prob,
    axis=1
)

prob_true, prob_pred = calibration_curve(
    correct,
    confidence,
    n_bins=10,
    strategy="uniform"
)

plt.figure(figsize=(7, 6))

plt.plot(
    prob_pred,
    prob_true,
    marker="o",
    label="DMS Model"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)

plt.xlabel(
    "Mean Predicted Confidence"
)

plt.ylabel(
    "Observed Accuracy"
)

plt.title(
    "Reliability Diagram — DMS Model"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    REPORTS / "calibration_reliability.png",
    dpi=300
)

plt.show()

### KPI5 Explainability

In [ ]:
if hasattr(
    best_model,
    "feature_importances_"
):

    feature_importance = pd.DataFrame({

        "Feature": X_test.columns,

        "Importance":
            best_model.feature_importances_

    }).sort_values(
        "Importance",
        ascending=False
    )

else:

    feature_importance = pd.DataFrame()

feature_importance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(
    figsize=(10, 7)
)

sns.barplot(
    data=feature_importance,
    x="Importance",
    y="Feature"
)

plt.title(
    "Feature Importance of Frozen DMS Model"
)

plt.xlabel(
    "Importance"
)

plt.ylabel(
    "Feature"
)

plt.tight_layout()

plt.savefig(
    REPORTS / "feature_importance.png",
    dpi=300
)

plt.show()

### KPI6 Fairness

In [ ]:
fairness_result = {

    "Status": "Not Assessable",

    "Reason":
        "The selected DMS dataset does not contain "
        "sufficient demographic/protected subgroup "
        "attributes for fairness evaluation."

}

fairness_result

### KPI7 Privacy

In [ ]:
privacy_result = {

    "Status": "Not Assessable",

    "Reason":
        "The current dataset does not provide the "
        "required privacy attack, identity leakage, "
        "or PII exposure ground truth."

}

privacy_result

## AI Trust Score

In [ ]:
WEIGHTS = {

    "Performance": 0.30,

    "Safety": 0.35,

    "Robustness": 0.20,

    "Calibration": 0.15

}

In [ ]:
trust_score = (

    performance_score
    * WEIGHTS["Performance"]

    +

    safety_score
    * WEIGHTS["Safety"]

    +

    robustness_score
    * WEIGHTS["Robustness"]

    +

    calibration_score
    * WEIGHTS["Calibration"]

)

trust_score_100 = (
    trust_score * 100
)

print(
    f"AI Trust Score: "
    f"{trust_score_100:.2f} / 100"
)

In [ ]:
trust_components = pd.DataFrame({

    "Dimension": [
        "Performance",
        "Safety",
        "Robustness",
        "Calibration"
    ],

    "Score": [
        performance_score,
        safety_score,
        robustness_score,
        calibration_score
    ],

    "Weight": [
        WEIGHTS["Performance"],
        WEIGHTS["Safety"],
        WEIGHTS["Robustness"],
        WEIGHTS["Calibration"]
    ]

})

trust_components["Weighted Contribution"] = (
    trust_components["Score"]
    *
    trust_components["Weight"]
)

trust_components

# RQ4 — KPI Contribution and Sensitivity Analysis

### Research Question 4
**Which KPI dimensions contribute most strongly to the overall AI Trust Score, and how sensitive is the composite score to changes in KPI values and weighting assumptions?**

This section extends the existing Trust Score analysis without changing the RQ1–RQ3 calculations. It separates:
1. baseline weighted contribution,
2. marginal KPI sensitivity,
3. ±0.10 KPI perturbation,
4. alternative weighting scenarios, and
5. KPI influence ranking.

**Important:** the analysis evaluates the proposed scoring framework. It does not claim that the researcher-defined weights are universally optimal.


In [ ]:
# RQ4.1 — Baseline weighted contribution

rq4_contribution = trust_components.copy()

rq4_contribution["Contribution_Percent"] = (
    rq4_contribution["Weighted Contribution"] * 100
)

rq4_contribution = rq4_contribution.sort_values(
    "Weighted Contribution",
    ascending=False
).reset_index(drop=True)

print("Baseline Trust Score:", f"{trust_score_100:.2f} / 100")
display(rq4_contribution)


In [ ]:
# RQ4.2 — Marginal sensitivity

# For TS = sum(w_i * M_i), d(TS)/d(M_i) = w_i.
sensitivity_summary = trust_components[
    ["Dimension", "Score", "Weight", "Weighted Contribution"]
].copy()

sensitivity_summary["Marginal Sensitivity"] = (
    sensitivity_summary["Weight"]
)

sensitivity_summary["Trust Score Change for ±0.10 KPI"] = (
    sensitivity_summary["Weight"] * 0.10 * 100
)

sensitivity_summary = sensitivity_summary.rename(
    columns={"Dimension": "KPI"}
).sort_values(
    "Marginal Sensitivity",
    ascending=False
).reset_index(drop=True)

display(sensitivity_summary)


In [ ]:
# RQ4.3 — One-at-a-time ±0.10 KPI perturbation

perturbation_rows = []

for kpi in KPI_SCORES:
    for delta in [-0.10, 0.10]:
        perturbed_scores = KPI_SCORES.copy()
        perturbed_scores[kpi] = np.clip(
            perturbed_scores[kpi] + delta,
            0,
            1
        )

        perturbed_trust_score = sum(
            perturbed_scores[name] * WEIGHTS[name]
            for name in KPI_SCORES
        )

        perturbation_rows.append({
            "KPI": kpi,
            "Change": delta,
            "Original KPI": KPI_SCORES[kpi],
            "Perturbed KPI": perturbed_scores[kpi],
            "Trust Score": perturbed_trust_score * 100,
            "Change in Trust Score": (
                perturbed_trust_score - trust_score
            ) * 100
        })

rq4_perturbation = pd.DataFrame(perturbation_rows)

display(rq4_perturbation)


In [ ]:
# RQ4.4 — Alternative weighting scenarios

weight_scenarios = {
    "Baseline": {
        "Performance": 0.30,
        "Safety": 0.35,
        "Robustness": 0.20,
        "Calibration": 0.15
    },
    "Equal weights": {
        "Performance": 0.25,
        "Safety": 0.25,
        "Robustness": 0.25,
        "Calibration": 0.25
    },
    "Safety emphasis": {
        "Performance": 0.25,
        "Safety": 0.45,
        "Robustness": 0.20,
        "Calibration": 0.10
    },
    "Performance emphasis": {
        "Performance": 0.40,
        "Safety": 0.25,
        "Robustness": 0.20,
        "Calibration": 0.15
    },
    "Robustness emphasis": {
        "Performance": 0.25,
        "Safety": 0.30,
        "Robustness": 0.30,
        "Calibration": 0.15
    },
    "Calibration emphasis": {
        "Performance": 0.25,
        "Safety": 0.30,
        "Robustness": 0.15,
        "Calibration": 0.30
    }
}

scenario_rows = []

for scenario, weights in weight_scenarios.items():
    assert np.isclose(sum(weights.values()), 1.0)

    scenario_score = sum(
        KPI_SCORES[kpi] * weights[kpi]
        for kpi in KPI_SCORES
    )

    scenario_rows.append({
        "Scenario": scenario,
        "Trust Score": scenario_score * 100,
        **{f"{kpi} Weight": weights[kpi] for kpi in KPI_SCORES}
    })

rq4_weight_sensitivity = pd.DataFrame(scenario_rows)

display(rq4_weight_sensitivity)


In [ ]:
# RQ4.5 — Visualisations

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    rq4_contribution["Dimension"],
    rq4_contribution["Weighted Contribution"] * 100
)
ax.set_ylabel("Contribution to Trust Score (percentage points)")
ax.set_xlabel("KPI Dimension")
ax.set_title("RQ4 — Baseline KPI Contributions")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(
    REPORTS / "rq4_kpi_contributions.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    sensitivity_summary["KPI"],
    sensitivity_summary["Trust Score Change for ±0.10 KPI"]
)
ax.set_ylabel("Absolute Trust Score Change (points)")
ax.set_xlabel("KPI")
ax.set_title("RQ4 — Marginal KPI Sensitivity")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(
    REPORTS / "rq4_kpi_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    rq4_weight_sensitivity["Scenario"],
    rq4_weight_sensitivity["Trust Score"]
)
ax.set_xlabel("AI Trust Score")
ax.set_ylabel("Weighting Scenario")
ax.set_title("RQ4 — Trust Score Sensitivity to Weighting Assumptions")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(
    REPORTS / "rq4_weight_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# RQ4.6 — Influence ranking

rq4_influence = sensitivity_summary[
    ["KPI", "Weight", "Marginal Sensitivity"]
].copy()

rq4_influence["Influence Rank"] = (
    rq4_influence["Marginal Sensitivity"]
    .rank(method="min", ascending=False)
    .astype(int)
)

display(rq4_influence)


### RQ4 interpretation

Under the baseline weighting scheme, the KPI with the largest weighted contribution is the largest contributor to the current Trust Score. Because the Trust Score is a weighted linear combination, the marginal sensitivity of each KPI is equal to its assigned weight.

Therefore, a 0.10 deterioration in a KPI changes the Trust Score by 0.10 × its weight. The alternative weighting analysis tests whether the overall Trust Score is stable under reasonable changes to the researcher-defined weights.

The results should be interpreted as evidence about the behaviour of the **proposed Trust Score framework**, rather than evidence that one trustworthiness dimension is universally more important than another.


In [ ]:
# RQ4.7 — Export all RQ4 results

rq4_contribution.to_csv(
    OUTPUT_DIR / "rq4_kpi_contributions.csv",
    index=False
)

sensitivity_summary.to_csv(
    OUTPUT_DIR / "rq4_sensitivity_summary.csv",
    index=False
)

rq4_perturbation.to_csv(
    OUTPUT_DIR / "rq4_kpi_perturbation.csv",
    index=False
)

rq4_weight_sensitivity.to_csv(
    OUTPUT_DIR / "rq4_weight_sensitivity.csv",
    index=False
)

rq4_influence.to_csv(
    OUTPUT_DIR / "rq4_influence_ranking.csv",
    index=False
)

print("RQ4 analysis and exports completed successfully.")


In [ ]:
robustness_results.to_csv(
    OUTPUT_DIR /
    "robustness_results.csv",
    index=False
)

trust_components.to_csv(
    OUTPUT_DIR /
    "ai_trust_score_components.csv",
    index=False
)

feature_importance.to_csv(
    OUTPUT_DIR /
    "explainability_feature_importance.csv",
    index=False
)

#print("✅ KPI results saved")
print("✅ Robustness results saved")
print("✅ Trust Score components saved")
print("✅ Explainability results saved")

## RQ4 — Final reporting statement

The RQ4 analysis identifies the relative contribution and sensitivity of the measurable KPI dimensions in the proposed AI Trust Score. Baseline contribution is calculated as \(w_iM_i\), while local sensitivity is assessed through the marginal effect \(\partial TS/\partial M_i=w_i\). A ±0.10 perturbation demonstrates the practical effect of KPI deterioration/improvement, and alternative weighting scenarios test the robustness of the composite score to researcher-defined weighting assumptions.

**Reporting caution:** the analysis demonstrates the behaviour and sensitivity of the proposed scoring framework; it does not establish universal causal importance of any trustworthiness dimension.
